# Install the packages

In [ ]:
import casparser
import pandas as pd
import requests
from pyxirr import xirr
 
PDF, PASSWORD = "cas_statement.pdf", ""   # PDF is unlocked, so no password
today = pd.Timestamp.today().normalize()
 
cas = casparser.read_cas_pdf(PDF, PASSWORD)

In [ ]:
from plotnine import *

In [ ]:
cas = cas.model_dump() if hasattr(cas, "model_dump") else cas

In [ ]:
cas

# Flatten into one table: one row per transaction

In [ ]:
# ---- Flatten into one table: one row per transaction ----
txns, nav, nav_date = [], {}, {}
for folio in cas["folios"]:
    for s in folio["schemes"]:
        key = (folio["folio"], s["scheme"])                       # the same fund in two folios is kept separate
        nav[key] = float(s["valuation"]["nav"])                   # latest NAV, used to value the units still held
        nav_date[key] = pd.Timestamp(s["valuation"]["date"])      # date of that NAV, used as the "sell today" date
        for t in s["transactions"]:
            # rows without units (stamp duty, STT, TDS) don't change holdings, so skip them
            if t["units"] is not None:
                kind = str(getattr(t["type"], "value", t["type"]))
                amt = float(t["amount"] or 0)
                # reinvested dividends were paid out of the fund's own NAV, not your pocket, so they cost nothing
                if kind == "DIVIDEND_REINVEST":
                    amt = 0.0
                txns.append((key, pd.Timestamp(t["date"]), kind, float(t["units"]), amt))
txns = pd.DataFrame(txns, columns=["key", "date", "kind", "units", "amount"])

In [ ]:
txns

# Work out which purchases are still held, fund by fund

In [ ]:
# ---- Work out which purchases are still held, fund by fund ----
rows = []
for key, g in txns.groupby("key"):
    g = g.sort_values("date")
    # positive units = money in (purchases, SIPs, switch-ins)
    buys = g[g.units > 0].copy()
 
    # a reversal (failed SIP, bounced payment) cancels one specific purchase,
    # so remove that purchase instead of treating the reversal as a sale
    for i, r in g[g.kind == "REVERSAL"].iterrows():
        match = buys[(buys.date <= r.date) & ((buys.units + r.units).abs() < 0.001)]
        if len(match):
            buys = buys.drop(match.index[-1])
            g = g.drop(i)
 
    # negative units = units gone (redemptions, switch-outs); total them up
    sold = -g.loc[g.units < 0, "units"].sum()
 
    # FIFO: sold units are taken from the oldest purchases first.
    # Running total of units bought, minus units sold, tells us how much
    # of each purchase is left (never below 0, never above what was bought).
    buys["held"] = (buys.units.cumsum() - sold).clip(lower=0).clip(upper=buys.units)
    buys = buys[buys.held > 0.001]   # drop purchases that were fully sold
 
    for d, u, a, h in zip(buys.date, buys.units, buys.amount, buys.held):
        # put each purchase in a bucket based on how long ago it was bought
        age = (today - d).days / 365.25
        b = "<1Y" if age < 1 else "1-3Y" if age < 3 else "3-5Y" if age < 5 else "5Y+"
        # two cash flows per purchase:
        #   money out on the purchase date (only for the share of units still held)
        #   money in on the NAV date = units still held x latest NAV
        #   the fund name is kept so non-equity funds can be filtered out later
        rows += [(b, d, -a * h / u, key[1]), (b, nav_date[key], h * nav[key], key[1])]
 

# XIRR for each bucket

In [ ]:
# ---- XIRR for each bucket ----
ORDER = ["5Y+", "3-5Y", "1-3Y", "<1Y"]

def summarise(cf):
    # invested, current value, XIRR and simple gain for each bucket
    return cf.groupby("bucket").apply(lambda x: pd.Series({
        "invested": -x.amount[x.amount < 0].sum(),   # cost of units still held
        "value": x.amount[x.amount > 0].sum(),       # what those units are worth today
        "xirr_%": 100 * xirr(x.date, x.amount),      # annualised return for the bucket
        "abs_%": 100 * (x.amount[x.amount > 0].sum() / -x.amount[x.amount < 0].sum() - 1),   # simple gain, no annualising (use this for <1Y)
    }), include_groups=False).reindex(ORDER).dropna(how="all")   # fixed order, skip empty buckets

cf = pd.DataFrame(rows, columns=["bucket", "date", "amount", "fund"])
print(summarise(cf).round(1))

# Nifty 50 benchmark: same money, same dates, into an index fund

In [ ]:
# ---- Load the benchmark's daily NAV history ----
BENCH = 120716   # AMFI code for UTI Nifty 50 Index Fund - Direct Plan - Growth

# free AMFI NAV data via mfapi.in
idx = pd.DataFrame(requests.get(f"https://api.mfapi.in/mf/{BENCH}", timeout=30).json()["data"])
idx["date"] = pd.to_datetime(idx.date, format="%d-%m-%Y").astype("datetime64[ns]")
idx["nav"] = idx.nav.astype(float)
idx = idx.sort_values("date")
last = idx.iloc[-1]   # latest index fund NAV, used to value the "what if" units
print(f"Benchmark NAV history: {idx.date.min():%d-%b-%Y} to {last.date:%d-%b-%Y}")

pd.set_option("display.width", 200)          # show all columns on one line
pd.set_option("display.max_columns", None)


def vs_nifty(cf):
    """Your bucket returns next to what the same purchases would have earned in the Nifty 50 fund."""
    # every purchase, with the index fund's NAV on that day
    # (or the last trading day before it, for weekends and holidays)
    buys = cf[cf.amount < 0].astype({"date": "datetime64[ns]"}).sort_values("date")   # same date format as idx, needed for the join
    buys = pd.merge_asof(buys, idx, on="date")
    if buys.nav.isna().any():
        raise ValueError(f"Some purchases are older than the benchmark's NAV history (starts {idx.date.min():%d-%b-%Y})")

    buys["idx_units"] = -buys.amount / buys.nav   # index units the same money would have bought

    # same two cash flows as before: money out on each purchase date, index units x latest NAV at the end
    bench_cf = pd.concat([
        buys[["bucket", "date", "amount"]],
        buys.groupby("bucket").idx_units.sum().mul(last.nav).rename("amount").reset_index().assign(date=last.date),
    ])

    mine, nifty = summarise(cf), summarise(bench_cf)
    return pd.DataFrame({
        "invested": mine.invested,
        "my_value": mine.value,
        "nifty_value": nifty.value,
        "my_xirr_%": mine["xirr_%"],
        "nifty_xirr_%": nifty["xirr_%"],
        "xirr_gap_%": mine["xirr_%"] - nifty["xirr_%"],   # positive = your funds beat the index
        "my_abs_%": mine["abs_%"],
        "nifty_abs_%": nifty["abs_%"],
    }).round(1)

# Compare: all funds

In [ ]:
print(vs_nifty(cf))

# Compare: equity funds only
Liquid, debt and gold/silver funds aren't meant to beat the Nifty, so they are left out here.

In [ ]:
# words in a fund's name that mark it as non-equity (debt, gold/silver, hybrid and similar).
# Built from AMFI's scheme categories; it matches over 99% of non-equity fund names in the
# public AMFI list and wrongly catches about 0.5% of equity ones (mostly closed-ended series).
NON_EQUITY = [
    # debt: very short to short term
    "Liquid", "Overnight", "Money Market", "Ultra Short", "Low Duration", "Short Duration",
    "Short Term Fund", "Savings Fund", "Cash", "Reserve", "Floater", "Floating Rate",
    "Floating Interest", "Treasury", "T-Bill", "Sovereign",
    # debt: medium to long term, credit, government, fixed term
    "Medium Duration", "Medium Term Fund", "Medium to Long", "Long Duration", "Long Term Fund",
    "Dynamic Bond", "Dynamic Term", "Dynamic Accrual", "Accrual", "Corporate Bond", "Credit Risk",
    "Credit Opportunities", "Banking and PSU", "Banking & PSU", "PSU Debt", "Gilt", "G-Sec",
    "Government Securities", "Constant Maturity", "SDL", "State Development", "Target Maturity",
    "Fixed Maturity", "Fixed Term", "Fixed Tenure", "FMP", "FIIF", "FTIF", "Interval",
    "Capital Protection", "Capital Preservation", "IBX", "Bond", "Debt", "Income", "Fixed Income",
    "Regular Income", "Monthly Income", "MIP", "Yield Plus", "Multiple Yield", "Fixed Horizon",
    "Regular Saving", "Charity",
    # commodities
    "Gold", "Silver",
    # hybrid and mixed
    "Hybrid", "Balanced", "Balanced Advantage", "Asset Allocation", "Multi Asset", "Multi-Asset",
    "Dynamic Asset", "Equity Savings", "Arbitrage", "Conservative Hybrid",
    # solution oriented (retirement, children)
    "Retirement", "Pension", "Children", "Child Care", "Life Cycle", "Lifecycle",
]

# whole-word match, so "Gold" does not catch "Goldman"; spaces in a phrase match any run of spaces
NON_EQUITY_RE = "|".join(r"(?<![A-Za-z])" + w.replace(" ", r"\s+") + r"(?![A-Za-z])" for w in NON_EQUITY)

equity_cf = cf[~cf.fund.str.contains(NON_EQUITY_RE, case=False, regex=True)]
print("Funds included:", *sorted(equity_cf.fund.unique()), sep="
  ")
print()
print(vs_nifty(equity_cf))

# Plots: my funds vs Nifty 50

In [ ]:
# ---- Plot: ggplot-style facet grid -> rows = metric (XIRR, absolute %), columns = holding-period bucket ----
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

BG, DARK, ACCENT, INDIA, MUTED, GRID = "#FAFAF7", "#1A1A1A", "#C0392B", "#2C5F8A", "#888888", "#E8E8E0"
PANEL, STRIP = "#EFEFE8", "#DCDCD3"   # ggplot-style: shaded panels, slightly darker strips
plt.rcParams["font.family"] = "serif"


def strip(ax, text, side="top"):
    """Grey label band on top of a panel (bucket) or on its right (metric), like ggplot facet strips."""
    if side == "top":
        ax.add_patch(mpatches.Rectangle((0, 1.0), 1, 0.11, transform=ax.transAxes, clip_on=False, fc=STRIP, ec="none"))
        ax.text(0.5, 1.055, text, transform=ax.transAxes, ha="center", va="center", fontsize=10, color=DARK)
    else:
        ax.add_patch(mpatches.Rectangle((1.0, 0), 0.09, 1, transform=ax.transAxes, clip_on=False, fc=STRIP, ec="none"))
        ax.text(1.045, 0.5, text, transform=ax.transAxes, ha="center", va="center", rotation=270, fontsize=10, color=DARK)


def plot_vs_nifty(t, label, fname):
    """Facet grid: one column per bucket, XIRR row on top, absolute return row below; my funds vs Nifty 50 in each panel."""
    t = t.reindex([b for b in ORDER if b in t.index])

    # headline: count buckets where my funds won (XIRR for 1Y+, absolute % for <1Y where XIRR misleads)
    won = sum((r["my_abs_%"] > r["nifty_abs_%"]) if b == "<1Y" else (r["xirr_gap_%"] > 0) for b, r in t.iterrows())
    title = f"{label} beat the Nifty 50 in {won} of {len(t)} buckets"

    rows = [("my_xirr_%", "nifty_xirr_%", "XIRR (% a year)"),
            ("my_abs_%", "nifty_abs_%", "Absolute return (%)")]

    # sharey="row": panels in a row share one y scale, like facet_grid(scales = "free_y")
    fig, axes = plt.subplots(2, len(t), figsize=(3 * len(t) + 1.5, 6.5), sharey="row", squeeze=False)
    fig.patch.set_facecolor(BG)

    for r, (mine_col, nifty_col, metric) in enumerate(rows):
        for c, (bucket, row) in enumerate(t.iterrows()):
            ax = axes[r, c]
            ax.set_facecolor(PANEL)   # shaded panel; the gaps between panels act as the dividers

            # XIRR under 1 year is annualised from a few months, so fade it
            fade = 0.35 if (bucket == "<1Y" and r == 0) else 1.0
            vals = [row[mine_col], row[nifty_col]]
            for x, v, colour in zip([0, 1], vals, [ACCENT, INDIA]):
                ax.bar(x, v, 0.6, color=colour, alpha=fade)
                # label above positive bars, below negative ones
                ax.text(x, v, f"{v:.1f}", ha="center", va="bottom" if v >= 0 else "top", fontsize=8.5,
                        color=colour, fontweight="bold" if colour == ACCENT else "normal", alpha=max(fade, 0.6))

            ax.axhline(0, color=DARK, lw=0.8)
            ax.grid(axis="y", color=BG, lw=1.0)   # light gridlines on the shaded panel, as in ggplot
            ax.set_axisbelow(True)
            for side in ["top", "right", "left", "bottom"]:
                ax.spines[side].set_visible(False)
            ax.tick_params(length=0, colors=DARK, labelsize=9)
            ax.set_xlim(-0.6, 1.6)
            ax.set_xticks([0, 1], ["My funds", "Nifty 50"] if r == 1 else ["", ""])
            ax.margins(y=0.2)   # room for the labels

            if r == 0:
                strip(ax, bucket, "top")
                if fade < 1:
                    ax.text(0.5, 0.97, "faded: under a year,\nread abs %", transform=ax.transAxes,
                            ha="center", va="top", fontsize=7.5, color=MUTED)
            if c == len(t) - 1:
                strip(ax, metric, "right")

    fig.suptitle(title, x=0.04, ha="left", fontsize=13, fontweight="bold", color=DARK)
    # fixed gaps between panels (wspace/hspace), like ggplot's panel.spacing
    fig.subplots_adjust(left=0.07, right=0.93, top=0.84, bottom=0.11, wspace=0.05, hspace=0.18)
    fig.text(0.04, 0.012, f"Source: CAMS/KFintech CAS · Nifty 50 = UTI Nifty 50 Index Fund Direct Growth (mfapi.in), same dates & amounts · as of {last.date:%d %b %Y}",
             fontsize=7.5, color=MUTED, ha="left")
    fig.savefig(fname, dpi=180, facecolor=BG)
    plt.show()

In [ ]:
plot_vs_nifty(vs_nifty(cf), "All funds", "all_funds_vs_nifty.png")

In [ ]:
plot_vs_nifty(vs_nifty(equity_cf), "Equity funds", "equity_funds_vs_nifty.png")

In [ ]:
# ---- Plot (plotnine = ggplot2 grammar in Python): share of money invested in each bucket ----
from plotnine import *

share = summarise(cf).invested
alloc = pd.DataFrame({"bucket": share.index, "pct": 100 * share.values / share.sum()})
alloc["bucket"] = pd.Categorical(alloc.bucket, categories=[b for b in ORDER[::-1] if b in share.index])  # oldest on top after coord_flip
alloc["top"] = alloc.pct == alloc.pct.max()   # biggest bucket gets the accent colour
alloc["label"] = alloc.pct.map("{:.1f}%".format)

top = alloc.loc[alloc.top].iloc[0]

p = (
    ggplot(alloc, aes("bucket", "pct", fill="top"))
    + geom_col(width=0.6)
    + geom_text(aes(label="label", colour="top"), ha="left", nudge_y=1, size=9, family="serif")
    + coord_flip()
    + scale_fill_manual(values={True: ACCENT, False: "#BBBBBB"}, guide=None)
    + scale_colour_manual(values={True: ACCENT, False: DARK}, guide=None)
    + scale_y_continuous(limits=(0, alloc.pct.max() * 1.15), expand=(0, 0))
    + labs(title=f"{top.bucket} holds {top.pct:.0f}% of the money invested",
           caption="Share of amount invested in units still held, by time since purchase · Source: CAMS/KFintech CAS")
    + theme_minimal(base_family="serif")
    + theme(
        figure_size=(8, 3.2),
        plot_background=element_rect(fill=BG, colour=BG),
        panel_background=element_rect(fill=BG, colour=BG),
        panel_grid=element_blank(),
        axis_title=element_blank(),
        axis_text_x=element_blank(),
        axis_text_y=element_text(colour=DARK, size=10),
        plot_title=element_text(face="bold", size=13, colour=DARK, ha="left"),
        plot_caption=element_text(size=7.5, colour=MUTED, ha="left"),
    )
)

#p.save("allocation_by_bucket.png", dpi=180, verbose=False)
p